# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described with a Croissant schema and is accessible via the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed (installs just for the current session)
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Print metadata
ds_meta = dataset.metadata
print(f"{ds_meta.name}: {ds_meta.description}")
print(f"Version: {getattr(ds_meta, 'version', 'N/A')}")
print(f"Published: {getattr(ds_meta, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets and fields. 
All referenced entities use their `@id` as unique identifiers.

In [ ]:
# List all record sets from the metadata (using their @id)
if hasattr(ds_meta, 'recordSets'):
    record_sets = ds_meta.recordSets
else:
    # Fallback to property name used in the Croissant schema
    record_sets = getattr(ds_meta, 'recordSet', [])

print("Record Sets and fields available:")
recordset_ids = []
for rs in record_sets:
    rs_id = getattr(rs, '@id', None)
    rs_name = getattr(rs, 'name', None)
    print(f"- RecordSet @id: {rs_id}, name: {rs_name}")
    recordset_ids.append(rs_id)
    if hasattr(rs, 'fields'):
        fields = rs.fields
    else:
        # Sometimes the field is called 'field' in the Croissant structure
        fields = getattr(rs, 'field', [])
    for field in fields:
        field_id = getattr(field, '@id', None)
        field_name = getattr(field, 'name', None)
        print(f"    - Field @id: {field_id}, name: {field_name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All lookups and selections will reference record sets and fields by their `@id`.

In [ ]:
# Collect all record set @ids (from previous overview)
# If none found, try to infer a common @id for demonstration, else error
if not recordset_ids:
    raise RuntimeError("No record sets found in metadata.")
else:
    # choose the first record set as example (update index for another)
    example_recordset_id = recordset_ids[0]

# Load records for the record set by @id
records = list(dataset.records(record_set=example_recordset_id))
df = pd.DataFrame(records)
print(f"Loaded DataFrame from RecordSet @id: {example_recordset_id}")
print("Columns:", df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
We'll filter, normalize, and group by columns using their `@id` as in the schema.

Below we demonstrate with a numeric field. Edit `numeric_field_id` and `group_field_id` to match a field's `@id` as per the printed list above, e.g. for 'coverage' or 'MW'.

In [ ]:
# Choose a numeric field's @id (update based on overview above)
# Example: Let's assume the field for 'coverage' percentage is 'coverage', or use the exact @id.
numeric_field_id = 'coverage'  # <-- replace by actual @id (e.g., 'cr:coverage')
# Choose a grouping field @id (e.g., 'accession')
group_field_id = 'accession'

# Check if the field is present
if numeric_field_id not in df.columns:
    print(f"Field '{numeric_field_id}' not found in columns: {df.columns.tolist()}")
else:
    # Drop NAs
    df_valid = df.dropna(subset=[numeric_field_id])
    # Try to coerce to numeric if not already
    df_valid[numeric_field_id] = pd.to_numeric(df_valid[numeric_field_id], errors='coerce')
    threshold = df_valid[numeric_field_id].median()  # for demo, use median as threshold
    filtered_df = df_valid[df_valid[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold} (using median as threshold):")
    print(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nFirst rows of normalized '{numeric_field_id}':")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id}, showing mean {numeric_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize the distribution and relationships between fields.

*Please set `numeric_field_id` and `group_field_id` as in the EDA above.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# Boxplot by group (if feasible)
if (numeric_field_id in df.columns) and (group_field_id in df.columns):
    plt.figure(figsize=(10,5))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.xticks(rotation=45)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load and process the FAIR² dataset using `mlcroissant`. Key steps included:
- Loading record sets and fields using their `@id`
- Filtering, normalizing, and grouping numeric fields
- Simple data visualizations to understand distributions

You may continue with more advanced analyses (e.g. statistical modeling or cross-record set joining) by referencing the `@id` of relevant fields and record sets following the same methodology as above.